In [3]:
{
  "cells": [
    {
      "cell_type": "code",
      "execution_count": null,
      "id": "24a9e2ba-6bac-4c90-8e1e-5b38749a88c2",
      "metadata": {
        "trusted": true
      },
      "outputs": [],
      "source": [
        "import random, time, tkinter as tk\n",
        "# random -> مكتبة بتخليك تختار عشوائي سواء كانت من ليستاية او رقم واستعملناها في اللابات\n",
        "# time -> بتحسب الوقت اللي بيعدي هنحتاجها في الواجهة \n",
        "# tkinter -> دي مكتبة بتاعت بايثون استخدمناها عشان نعمل الواجهة الاساسية لو محدش غيرها\n",
        "N, TIME_LIMIT, NUM_WORDS, MAX_EFFORT = 8, 300, 10, 1000\n",
        "# N -> grid size aka 8*8 , TIME_LIMIT -> literally time limit in seconds\n",
        "# NUM_WORDS ->  عدد الكلام المستعمل في اللعبة انا شايف انو 10 احسن حاجة عشان هتوضح ليه بنفضل ستايلات على ستايلات تانية \n",
        "# MAX_EFFORT -> اعلى عدد من محاولات انك تحط كلمة في اماكن مختلفة \n",
        "HINT_COST = 5   # points taken off for each hint\n",
        "POOL = [\"HOUSE\", \"WATER\", \"APPLE\", \"TABLE\", \"CHAIR\", \"MUSIC\", \"LIGHT\", \"PHONE\", \"BREAD\",\n",
        "        \"GREEN\", \"SMILE\", \"RIVER\", \"CLOUD\", \"PLANT\", \"TRAIN\", \"SUGAR\", \"MONEY\", \"HAPPY\",\n",
        "        \"SUMMER\", \"WINTER\", \"GARDEN\", \"FRIEND\", \"FAMILY\", \"SCHOOL\", \"WINDOW\", \"DOCTOR\",\n",
        "        \"ORANGE\", \"KITCHEN\", \"MORNING\", \"COUNTRY\", \"BIRTHDAY\", \"MOUNTAIN\", \"ANIMAL\"]\n",
        "# هنا زود كلام براحتك في الليستاية دي اهم حاجة الكلمة متعديش ال 8 حروف\n",
        "DIRS = [(0, 1), (1, 0), (1, 1), (-1, 1), (0, -1), (-1, 0), (-1, -1), (1, -1)]\n",
        "# دي الاتجهات هنحتاجها في المعادلة تحت على طول\n",
        "\n",
        "# Finds all valid grid positions for a word.\n",
        "def placements(grid, word):\n",
        "# هنا الفنكشن دي بتتاكد من كل الاماكن اللي كل حرف من الكلمة ينفع يتحط فيها لو المكان فيه نفس الحرف او فاضي هيعدي غير كدا هنروح للي بعدها\n",
        "    out = []\n",
        "    for r in range(N):\n",
        "        for c in range(N):\n",
        "            for dr, dc in DIRS:\n",
        "# السطر اللي تحت ده من الاخر بيعمل ليستاية جواها كل الاماكن اللي الكلمة ينفع تتحط فيها بيجرب كل الاتجاهات وكل عمود في كل صف \n",
        "# CAT مثال على كلمة \n",
        "# cat is 3 letters so index 0 1 2 , cells is a list which will have three tuples of placements for each letter of the word cat \n",
        "#let's say the index row in the loop is 3 and column is 2 and the direction is (1,-1) the loop underneath will make the cells = [(3,2),(4,1),(5,0)]\n",
        "#which if you look at the grid is a diagonal from right to left      \n",
        "                cells = [(r + dr * i, c + dc * i) for i in range(len(word))]\n",
        "                if all(0 <= y < N and 0 <= x < N and grid[y * N + x] in \".%s\" % ch\n",
        "                       for (y, x), ch in zip(cells, word)):\n",
        "# all makes sure that every condition is true\n",
        "# first zip(cells,word) makes it so that every placement is with it's character in the word like this ((3, 2), 'C'), ((4, 1), 'A'), ((5, 0), 'T') \n",
        "# so the if condition only makes sure that everything is true as in y \"row index\" is bigger or equal to 0 and smaller than the number of grid which is 8 so maximum index is 7\n",
        "# column is same as row and that the cell number since our grid is literally a big string of N*N we calculate the index number using y * N + x we make sure that it's either a . which is empty or the character that we are iterating through and if allat is True we append the cells to out\n",
        "                    out.append(cells)\n",
        "    return out\n",
        "\n",
        "# Puts a word's letters onto the grid.\n",
        "# هنا الفنكشن دي حرفيا بتحط الحروف في مكانها في الجريد\n",
        "def place(grid, word, cells):\n",
        "#  بتاخد كوبي من الجريد وبتحطها في ليست عشان تعدل عليها  \n",
        "    g = list(grid)\n",
        "    for (y, x), ch in zip(cells, word):\n",
        "        g[y * N + x] = ch\n",
        "# هنا ال ريتيرن بترجع الليستاية كا سترينج تاني بعد ما اتعدل عليها\n",
        "    return \"\".join(g)\n",
        "\n",
        "\n",
        "# دي التلت ستايلات اللي عندنا كل واحدة بتاخد الجريد والسيلز وبتشتغل عليهم \n",
        "STYLES = {\n",
        "    # overlap \n",
        "    # sort كل مكان في السيلز عبارة عن مكان على الجريد الفنكشن بترجع مجموع الاماكن اللي لقيتها مش فاضية وبتحط في الاخر سالب عشان فنشن ال \n",
        "    # تعتبرها اختيار افضل\n",
        "    \"overlap\": lambda grid, cells: -sum(grid[y * N + x] != \".\" for y, x in cells),\n",
        "    # بالنسبة لل edge\n",
        "    #   هنا بتفضل لما تبقى اماكن السيلز بعيدة عن النص بتاع الجريد مثال لو حسبت مجموع السيلز دي\n",
        "    # (0,0) , (0,1) , (0,2)\n",
        "    # outcome : 18 and with the outside negative it becomes -18 وزي ما قولت السالب بيخليها افضل لما نستعمل عليها فنكشن السورت\n",
        "    \"edge\":    lambda grid, cells: -sum(abs(y - (N - 1) / 2) + abs(x - (N - 1) / 2) for y, x in cells),\n",
        "    # هنا حرفيا عشوائية وكل ما كان الرقم اصغر كل ما فنكشن السورت اعتمدته الاول\n",
        "    \"random\":  lambda grid, cells: random.random(),\n",
        "}\n",
        "# هنا حرفيا ديكشنري لكل ستايل جواه ليست فاضية هتتملي بعدين بعدد المحاولات الستايل احتاجها عشان يعمل لعبة لما نستعملهم عشان نعرف اني واحدة اسهل واحسن \n",
        "history = {s: [] for s in STYLES} \n",
        "\n",
        "\n",
        "# Picks the best or random AI placement style.\n",
        "def pick_style():\n",
        "#هنا هيشوفهم واحد واحد قبل اي حاجة هيشوف لو الستايل في الديكشنري عبارة عن ليستاية فاضية هيجربه عشان مجربهوش قبل كدا\n",
        "    for s in STYLES:\n",
        "        if not history[s]:\n",
        "            return s\n",
        "#    فيه نسبة انو يجرب ستايل عشوائي لو الرقم العشوائي طلع اقل من 0.2 او نسبة 20% يعني\n",
        "    if random.random() < 0.2:                      \n",
        "        return random.choice(list(STYLES))\n",
        "#  لو عدا من كل حاجة من دي في العادي بيختار احسن ستايل بانه بيروح لكل واحد ويحسب عدد محاولاته على اشتغل كام مرة واقل رقم هيطلع احسن واحد فا هيختاره\n",
        "# example : history = {\n",
        "#    \"overlap\": [100, 200, 300],\n",
        "#    \"edge\": [50, 60, 70],\n",
        "#    \"random\": [400, 500, 600]\n",
        "#}     \n",
        "# (100 + 200 + 300) / 3 = 200 , (50 + 60 + 70) / 3 = 60 , (400 + 500 + 600) / 3 = 500\n",
        "# edge اقل رقم ال\n",
        "    return min(STYLES, key=lambda s: sum(history[s]) / len(history[s]))\n",
        "\n",
        "# exception هنحتاج ال   \n",
        "# عشان احنا في ال فنكشن اللي بعدها باكتراك هنقعد ننادي الفنكشن جوا نفسها كتير فا لما عدد المحاولات يعدي الماكس بتاعنا اللي هو كان 1000 عايزين نطلع من كل الفنكشنز اللي جوا بعض دي على طول\n",
        "class TooHard(Exception):\n",
        "    pass\n",
        "\n",
        "# دي زي وكانك عملت كل واحدة لوحدها عادي هي هي\n",
        "effort = backtracks = 0\n",
        "chosen = []          # NEW: cells of every word placed so far, in order (same order as words)\n",
        "\n",
        "# Recursively places all words onto the grid.\n",
        "# هنا بقى واحدة واحدة انا هشرحها كاملة من فوق هنا عشان مينفعش ادخل جواها هيبقى شكلها معقد\n",
        "# grid -> N*N string of \".\" , words -> list of randomly selected words , i which is index which is 0 , style -> chosen style\n",
        "# uses the same global variables that are outside the function\n",
        "# if i == len(words) that means the function recurred till the end with no issues \n",
        "# we get this possible placements for the current grid and the current word inside a variable named options and we shuffle them so it lowers the chance of getting the same puzzle twice\n",
        "# and then we sort them using the selected style and yes even if we sort them we still need the shuffle since if two placements have the same value the sort is gonna leave them as their original order\n",
        "# again options are like this options = [  [(0, 0), (0, 1), (0, 2)], [(2, 1), (2, 2), (2, 3)], [(4, 3), (4, 4), (4, 5)]  ] so cells in options are a list of tuples \n",
        "# now for every time we try a cell the effort += 1 so we can measure if it's taking too many attempts and it's not worth it and if we go over the max effort we raise the exception toohard which interrupts every function in the recursion and stops them\n",
        "# now for the bigger picture now let's say the first word is DOG the second is OWL now we go through every cell in options for DOG now let's say we go through the first one now the result awaits for another call of the function backtrack \n",
        "# but this time it's a new grid which is a copy of the original modified with the placement that we used for DOG and i + 1 so it goes to the next word which is OWL now again we see all the possible options for this word with the new grid which DOG is placed in already\n",
        "# now we try the same thing with OWL we go through all of it's possible cells now let's say we go through the first one the same cycle repeats with the next words but if the next word doesn't have a valid placement we go back one cycle now let's say we chose a placement for OWL and then the next cycle was FOX but FOX doesn't have any valid placement \n",
        "# then it returns None for the previous cycle which was OWL now owl's loop tries another placement and i think you get the idea by now now let's say it all worked out and the final word found a valid placement the new word placement is placed in the new copied grid it made and the first condition becomes True which then returns the final grid of all placed words the final grid keeps returning to the previous cycles till it returns to the very first one and returns the final grid\n",
        "# now again if the effort exceeds max effort we raise the too hard exception and all of this recursion is interrupted and it doesn't make any problems since every cycle edits it's own copied grid\n",
        "# also everytime a cycle returns None to it's previous cycle and it makes it's loop try another placement (cells in options) we add + 1 to the backtracks \n",
        "def backtrack(grid, words, i, style):\n",
        "    global effort, backtracks\n",
        "    if i == len(words):\n",
        "        return grid\n",
        "    options = placements(grid, words[i])\n",
        "    random.shuffle(options)\n",
        "    options.sort(key=lambda cells: STYLES[style](grid, cells))\n",
        "    for cells in options:\n",
        "        effort += 1\n",
        "        if effort > MAX_EFFORT:\n",
        "            raise TooHard\n",
        "        chosen.append(cells)                       # NEW: try this placement\n",
        "        result = backtrack(place(grid, words[i], cells), words, i + 1, style)\n",
        "        if result:\n",
        "            return result\n",
        "        chosen.pop()                               # NEW: it failed, undo it\n",
        "        backtracks += 1\n",
        "    return None\n",
        "\n",
        "\n",
        "# Generates a new complete word search puzzle.\n",
        "def make_puzzle():\n",
        "    # uses the same global effort , backtracks variables\n",
        "    global effort, backtracks\n",
        "    while True:\n",
        "    # using the pick_style function to pick a style\n",
        "        style = pick_style()\n",
        "    # picking random different words from the pool of words how many is indicated by the NUM_WORDS variable\n",
        "        words = random.sample(POOL, NUM_WORDS)\n",
        "    # long words first = fewer dead ends   \n",
        "        words.sort(key=len, reverse=True)  \n",
        "    # zeroing everything every loop\n",
        "        effort = backtracks = 0\n",
        "        chosen.clear()                              # NEW\n",
        "        try:\n",
        "    # here we try the backtrack function to make the grid of our game\n",
        "            grid = backtrack(\".\" * (N * N), words, 0, style)\n",
        "        except TooHard:\n",
        "    # and if we ever raise the toohard exception we catch it here and append to the list of the of the style key in the dictionary and we continue another round of the loop so in summary we try again\n",
        "            history[style].append(MAX_EFFORT)       # punish the style, then try again\n",
        "            continue\n",
        "    # here if we actually succeed and get our final grid with all the valid placements \n",
        "        if grid:\n",
        "    # we append the number of efforts it took to the list of the of the style key in the dictionary so that we can check if this is the most efficient when making another puzzle by it taking fewer attempts on average\n",
        "            history[style].append(effort)   \n",
        "            # here we just take the final grid and if there is an empty position which is indicated by \".\" we replace it with a random character and if not we leave the original character\n",
        "            grid = \"\".join(random.choice(\"ABCDEFGHIJKLMNOPQRSTUVWXYZ\") if ch == \".\" else ch\n",
        "                           for ch in grid)\n",
        "            # NEW: also return where every word was placed {word: cells}\n",
        "            return words, grid, style, effort, backtracks, dict(zip(words, chosen))\n",
        "\n",
        "\n",
        "# ---------------- GAME / GUI PART (CREAM EDITION) ----------------\n",
        "import tkinter.font as tkfont\n",
        "import math\n",
        "\n",
        "# ---- الألوان: كلها هنا وتقدر تغيّر أي واحد ----\n",
        "BG          = \"#faf7f1\"   # خلفية الشاشة (كريمي)\n",
        "TILE        = \"#ebe3d2\"   # الخانة العادية\n",
        "TILE_EDGE   = \"#d6cbb3\"\n",
        "TILE_HOVER  = \"#e2d8c3\"   # لما الماوس يعدي على خانة\n",
        "INK         = \"#2a2118\"   # لون الكلام\n",
        "MUTED       = \"#8a7f6f\"\n",
        "GREEN       = \"#6cbb8a\"   # الكلمة اللي اتلقت\n",
        "GREEN_DARK  = \"#4f9b6d\"\n",
        "ORANGE      = \"#f0a35e\"   # السحب\n",
        "BTN         = \"#d96a1f\"   # زرار New Game\n",
        "BTN_HOVER   = \"#c25a14\"\n",
        "HINT_FILL   = \"#fbe3c4\"   # الحرف الأول بتاع التلميح\n",
        "HINT_EDGE   = \"#e8944a\"\n",
        "MISS_FILL   = \"#f9d6d6\"   # الكلمات اللي متلقتش بعد ما الوقت يخلص\n",
        "MISS_EDGE   = \"#d64545\"\n",
        "PANEL       = \"#e6e9ef\"   # بانل الكلمات\n",
        "PANEL_EDGE  = \"#c8cfdb\"\n",
        "TRACK       = \"#e8e0d4\"\n",
        "NAVY        = \"#1f2937\"\n",
        "RED         = \"#d64545\"\n",
        "WHITE       = \"#ffffff\"\n",
        "\n",
        "# ألوان مختلفة لكل كلمة تتلقي\n",
        "WORD_COLORS = [\n",
        "    \"#57b880\",  #green\n",
        "    \"#4c9be8\",  #blue\n",
        "    \"#e06c75\",  #red\n",
        "    \"#9b72cf\",  #purple\n",
        "    \"#3aa89e\",  #teal\n",
        "    \"#d97b38\",  #orange\n",
        "    \"#d4609c\",  #pink\n",
        "    \"#6d8ce8\",  #sky blue\n",
        "    \"#82ad49\",  #lime\n",
        "    \"#c98a2c\",  #yellow\n",
        "]\n",
        "\n",
        "# ---- الفونت ----\n",
        "FONT = \"Segoe UI\"         # بدائل: \"Bahnschrift\", \"Trebuchet MS\"\n",
        "\n",
        "# ---- المقاسات ----\n",
        "T, G, PAD = 56, 6, 4      # حجم الخانة، المسافة بين الخانات، الهامش\n",
        "BOARD_W = N * T + (N - 1) * G + 2 * PAD\n",
        "CONTENT_W = 620           # عرض بانل الكلمات\n",
        "COLS = 5                  # عدد الكلمات في الصف\n",
        "\n",
        "root = tk.Tk()\n",
        "root.title(\"AI Word Search\")\n",
        "root.configure(bg=BG)\n",
        "root.resizable(False, False)\n",
        "\n",
        "F_CELL  = tkfont.Font(family=FONT, size=18, weight=\"bold\")\n",
        "F_WORD  = tkfont.Font(family=FONT, size=13)\n",
        "F_DONE  = tkfont.Font(family=FONT, size=13, overstrike=True)\n",
        "F_TITLE = tkfont.Font(family=FONT, size=10, weight=\"bold\")\n",
        "F_WIN   = tkfont.Font(family=FONT, size=40, weight=\"bold\")\n",
        "\n",
        "state = dict(words=[], grid=\"\", found=set(), found_lines=[], done={}, first=None,\n",
        "             preview=[], left=0, score=0, running=False, job=None, hover=None,\n",
        "             positions={}, hint=None, reveal=set(), bjob=None)\n",
        "\n",
        "\n",
        "# Draws a rounded rectangle on a canvas.\n",
        "def rrect(cv, x1, y1, x2, y2, r, **kw):\n",
        "    \"\"\"مستطيل بحواف مدورة على Canvas.\"\"\"\n",
        "    pts = [x1 + r, y1, x2 - r, y1, x2, y1, x2, y1 + r, x2, y2 - r, x2, y2,\n",
        "           x2 - r, y2, x1 + r, y2, x1, y2, x1, y2 - r, x1, y1 + r, x1, y1]\n",
        "    return cv.create_polygon(pts, smooth=True, **kw)\n",
        "\n",
        "\n",
        "# Creates a custom rounded clickable button.\n",
        "def rbutton(parent, text, command, bg, hover_bg, fg, w=130, h=40, r=10):\n",
        "    \"\"\"زرار بحواف مدورة.\"\"\"\n",
        "    cv = tk.Canvas(parent, width=w, height=h, bg=BG, highlightthickness=0, cursor=\"hand2\")\n",
        "    shape = rrect(cv, 1, 1, w - 1, h - 1, r, fill=bg, outline=bg)\n",
        "    cv.create_text(w / 2, h / 2, text=text, font=(FONT, 11, \"bold\"), fill=fg)\n",
        "    cv.bind(\"<Enter>\", lambda e: cv.itemconfig(shape, fill=hover_bg, outline=hover_bg))\n",
        "    cv.bind(\"<Leave>\", lambda e: cv.itemconfig(shape, fill=bg, outline=bg))\n",
        "    cv.bind(\"<Button-1>\", lambda e: command())\n",
        "    return cv\n",
        "\n",
        "\n",
        "# ---- الهيدر ----\n",
        "header = tk.Frame(root, bg=BG)\n",
        "header.pack(fill=\"x\", padx=24, pady=(18, 6))\n",
        "tk.Label(header, text=\"AI WORD SEARCH\", font=(FONT, 20, \"bold\"), bg=BG, fg=INK).pack(side=\"left\")\n",
        "tk.Label(header, text=\"●\", font=(FONT, 10), bg=BG, fg=INK).pack(side=\"left\", padx=8)\n",
        "rbutton(header, \"NEW GAME\", lambda: new_game(), BTN, BTN_HOVER, WHITE).pack(side=\"right\")\n",
        "hint_btn = tk.Label(header, text=\"HINT (-%d pts)\" % HINT_COST, font=(FONT, 11, \"bold\"), bg=BG, fg=MUTED,\n",
        "                    cursor=\"hand2\")\n",
        "hint_btn.pack(side=\"right\", padx=(0, 18))\n",
        "hint_btn.bind(\"<Button-1>\", lambda e: hint())\n",
        "\n",
        "# ---- الإحصائيات ----\n",
        "stats = tk.Frame(root, bg=BG)\n",
        "stats.pack(fill=\"x\", padx=24)\n",
        "timer_lbl = tk.Label(stats, font=(FONT, 16, \"bold\"), bg=BG, fg=INK)\n",
        "timer_lbl.pack(side=\"left\")\n",
        "score_lbl = tk.Label(stats, font=(FONT, 16, \"bold\"), bg=BG, fg=INK)\n",
        "score_lbl.pack(side=\"right\")\n",
        "wordlabel = tk.Label(stats, font=(FONT, 12), bg=BG, fg=MUTED)\n",
        "wordlabel.pack(expand=True)\n",
        "\n",
        "bar = tk.Canvas(root, height=5, bg=TRACK, highlightthickness=0)\n",
        "bar.pack(fill=\"x\", padx=24, pady=(8, 0))\n",
        "\n",
        "msg = tk.Label(root, font=(FONT, 10, \"bold\"), bg=BG, fg=INK)\n",
        "msg.pack(pady=(10, 0))\n",
        "\n",
        "# ---- اللوحة ----\n",
        "board = tk.Canvas(root, width=BOARD_W, height=BOARD_W, bg=BG, highlightthickness=0)\n",
        "board.pack(pady=10)\n",
        "\n",
        "\n",
        "# Calculates the height needed for the word panel.\n",
        "def panel_h(n):\n",
        "    return 46 + ((n + COLS - 1) // COLS) * 30 + 10\n",
        "\n",
        "\n",
        "# ---- بانل الكلمات (تحت اللوحة) ----\n",
        "words_cv = tk.Canvas(root, width=CONTENT_W, height=panel_h(NUM_WORDS), bg=BG,\n",
        "                     highlightthickness=0)\n",
        "words_cv.pack(padx=24, pady=(0, 8))\n",
        "\n",
        "note = tk.Label(root, font=(FONT, 10), bg=BG, fg=RED)\n",
        "note.pack()\n",
        "statlabel = tk.Label(root, font=(FONT, 9), bg=BG, fg=\"#5c5245\")\n",
        "statlabel.pack(pady=(0, 14))\n",
        "\n",
        "\n",
        "# Gets the center screen coordinates of a cell.\n",
        "def center(r, c):\n",
        "    return PAD + c * (T + G) + T / 2, PAD + r * (T + G) + T / 2\n",
        "\n",
        "\n",
        "# Draws a colored highlight bar over a word.\n",
        "def draw_pill(cells, color):\n",
        "    \"\"\"شريط مدور بيغطي الكلمة من أول خانة لآخر خانة (في أي اتجاه).\"\"\"\n",
        "    (r0, c0), (r1, c1) = cells[0], cells[-1]\n",
        "    x0, y0 = center(r0, c0)\n",
        "    x1, y1 = center(r1, c1)\n",
        "    board.create_line(x0, y0, x1, y1, width=T - 4, fill=color, capstyle=\"round\", tags=\"b\")\n",
        "\n",
        "\n",
        "# Redraws all tiles, highlights, and letters on board.\n",
        "def draw_board(hint_cell):\n",
        "    board.delete(\"b\")\n",
        "    for r in range(N):\n",
        "        for c in range(N):\n",
        "            pos = (r, c)\n",
        "            x, y = PAD + c * (T + G), PAD + r * (T + G)\n",
        "            fill, edge = TILE, TILE_EDGE\n",
        "            if state[\"hover\"] == pos and state[\"running\"]: fill = TILE_HOVER\n",
        "            if pos in state[\"reveal\"]: fill, edge = MISS_FILL, MISS_EDGE   # NEW: missed words after time's up\n",
        "            if pos == hint_cell: fill, edge = HINT_FILL, HINT_EDGE         # NEW: hint\n",
        "            if state[\"first\"] == pos: fill, edge = ORANGE, ORANGE\n",
        "            rrect(board, x, y, x + T, y + T, 10, fill=fill, outline=edge, tags=\"b\")\n",
        "    for cells, color in state[\"found_lines\"]:\n",
        "        draw_pill(cells, color)\n",
        "    if len(state[\"preview\"]) > 1:                      # live drag highlight\n",
        "        draw_pill(state[\"preview\"], ORANGE)\n",
        "    for k, ch in enumerate(state[\"grid\"]):\n",
        "        pos = (k // N, k % N)\n",
        "        covered = pos in state[\"found\"] or pos in state[\"preview\"] or state[\"first\"] == pos\n",
        "        x, y = center(*pos)\n",
        "        board.create_text(x, y, text=ch, font=F_CELL, fill=WHITE if covered else INK, tags=\"b\")\n",
        "    board.tag_raise(\"win\")                             # شاشة الفوز دايما فوق\n",
        "\n",
        "\n",
        "# Displays the target words list with their statuses.\n",
        "def draw_words():\n",
        "    n = len(state[\"words\"]) or NUM_WORDS\n",
        "    W, H = CONTENT_W, panel_h(n)\n",
        "    words_cv.config(height=H)\n",
        "    words_cv.delete(\"all\")\n",
        "    rrect(words_cv, 2, 2, W - 2, H - 2, 14, fill=PANEL, outline=PANEL_EDGE)\n",
        "    words_cv.create_text(W / 2, 24, text=\"WORDS\", font=F_TITLE, fill=MUTED)\n",
        "    ptxt = \"\".join(state[\"grid\"][y * N + x] for y, x in state[\"preview\"])\n",
        "    for i, w in enumerate(sorted(state[\"words\"])):\n",
        "        x, y = W * ((i % COLS) + 0.5) / COLS, 56 + (i // COLS) * 30\n",
        "        if w in state[\"done\"]:\n",
        "            fnt, col = F_DONE, state[\"done\"][w]\n",
        "        elif ptxt and ptxt in (w, w[::-1]):            # الكلمة اللي بتسحبها دلوقتي\n",
        "            fnt, col = F_WORD, HINT_EDGE\n",
        "        else:\n",
        "            fnt, col = F_WORD, INK\n",
        "        words_cv.create_text(x, y, text=w, font=fnt, fill=col)\n",
        "\n",
        "\n",
        "# Updates the board, word list, timer, and score.\n",
        "def refresh():\n",
        "    # NEW: the hint is only shown while its word is still not found\n",
        "    hint_cell = None\n",
        "    if state[\"hint\"] and state[\"hint\"][0] not in state[\"done\"]:\n",
        "        hint_cell = state[\"hint\"][1]\n",
        "    draw_board(hint_cell)\n",
        "    draw_words()\n",
        "    low = state[\"left\"] <= 30\n",
        "    can_hint = state[\"running\"] and state[\"score\"] >= HINT_COST\n",
        "    wordlabel.config(text=\"%d / %d found\" % (len(state[\"done\"]), len(state[\"words\"])))\n",
        "    timer_lbl.config(text=\"%d:%02d\" % divmod(state[\"left\"], 60), fg=RED if low else INK)\n",
        "    score_lbl.config(text=\"%d pts\" % state[\"score\"])\n",
        "    hint_btn.config(text=\"HINT (-%d pts)\" % HINT_COST,\n",
        "                    fg=MUTED if can_hint else \"#c9c0b0\")\n",
        "    bar.delete(\"all\")                                  # الوقت اللي عدّى بيكبر من اليمين\n",
        "    w = bar.winfo_width()\n",
        "    used = (TIME_LIMIT - state[\"left\"]) / TIME_LIMIT\n",
        "    bar.create_rectangle(w * (1 - used), 0, w, 5, fill=RED if low else NAVY, width=0)\n",
        "\n",
        "\n",
        "# Resets everything and starts a fresh game.\n",
        "def new_game():\n",
        "    if state[\"job\"]:\n",
        "        root.after_cancel(state[\"job\"])\n",
        "    if state[\"bjob\"]:\n",
        "        root.after_cancel(state[\"bjob\"])\n",
        "        state[\"bjob\"] = None\n",
        "    board.delete(\"win\")\n",
        "    note.config(text=\"\")\n",
        "    t = time.time()\n",
        "    words, grid, style, tries, undone, positions = make_puzzle()\n",
        "    state.update(words=words, grid=grid, found=set(), found_lines=[], done={},\n",
        "                 first=None, preview=[], left=TIME_LIMIT, running=True,\n",
        "                 positions=positions, hint=None, reveal=set(), score=0)\n",
        "    msg.config(text=\"AI used '%s': %d attempts, %d backtracks, %.2fs\"\n",
        "               % (style, tries, undone, time.time() - t), fg=INK)\n",
        "    statlabel.config(text=\"AI's avg attempts (lower = better):  \" + \"   \".join(\n",
        "        \"%s %.0f\" % (s, sum(v) / len(v)) for s, v in history.items() if v))\n",
        "    refresh()\n",
        "    tick()\n",
        "\n",
        "\n",
        "# Counts down the timer and ends game at zero.\n",
        "def tick():\n",
        "    if not state[\"running\"]:\n",
        "        return\n",
        "    if state[\"left\"] <= 0:\n",
        "        state[\"running\"] = False\n",
        "        # NEW: reveal every word the player didn't find\n",
        "        for w in state[\"words\"]:\n",
        "            if w not in state[\"done\"]:\n",
        "                state[\"reveal\"].update(state[\"positions\"][w])\n",
        "        state[\"hint\"] = None\n",
        "        msg.config(text=\"TIME'S UP - YOU LOSE\", fg=RED)\n",
        "        note.config(text=\"The words you missed are shown in red\")\n",
        "        refresh()\n",
        "        return\n",
        "    state[\"left\"] -= 1\n",
        "    refresh()\n",
        "    state[\"job\"] = root.after(1000, tick)\n",
        "\n",
        "\n",
        "# Highlights the first letter of an unfound word.\n",
        "def hint():\n",
        "    # NEW: highlights the first letter of a random unfound word\n",
        "    if not state[\"running\"]:\n",
        "        return\n",
        "    if state[\"score\"] < HINT_COST:\n",
        "        note.config(text=\"Not enough points for a hint (need %d pts)\" % HINT_COST)\n",
        "        root.after(1500, lambda: note.config(text=\"\"))\n",
        "        return\n",
        "    left = [w for w in state[\"words\"] if w not in state[\"done\"]]\n",
        "    if not left:\n",
        "        return\n",
        "    w = random.choice(left)\n",
        "    state[\"hint\"] = (w, state[\"positions\"][w][0])\n",
        "    state[\"score\"] -= HINT_COST\n",
        "    note.config(text=\"Hint: a %d-letter word starts at the highlighted cell\" % len(w))\n",
        "    refresh()\n",
        "\n",
        "\n",
        "# Gets the grid row and column under the mouse.\n",
        "def cell_at(e):\n",
        "    \"\"\"Row, col of the cell under the mouse (or None if outside the board).\"\"\"\n",
        "    c, r = (e.x - PAD) // (T + G), (e.y - PAD) // (T + G)\n",
        "    return (int(r), int(c)) if 0 <= r < N and 0 <= c < N else None\n",
        "\n",
        "\n",
        "# Returns all cells along a straight drag line.\n",
        "def line(start, end):\n",
        "    \"\"\"Cells on the straight line from start to end ([] if it isn't straight).\"\"\"\n",
        "    (r0, c0), (r, c) = start, end\n",
        "    dy, dx = r - r0, c - c0\n",
        "    if dy != 0 and dx != 0 and abs(dy) != abs(dx):\n",
        "        return []\n",
        "    n = max(abs(dy), abs(dx)) + 1\n",
        "    sy, sx = (dy > 0) - (dy < 0), (dx > 0) - (dx < 0)\n",
        "    return [(r0 + sy * i, c0 + sx * i) for i in range(n)]\n",
        "\n",
        "\n",
        "# Starts word selection when the mouse is pressed.\n",
        "def press(r, c):\n",
        "    if state[\"running\"]:\n",
        "        state[\"first\"] = (r, c)          # drag starts here\n",
        "        refresh()\n",
        "\n",
        "\n",
        "# Updates the live highlight path while dragging.\n",
        "def motion(e):\n",
        "    if state[\"running\"] and state[\"first\"] is not None:\n",
        "        end = cell_at(e)\n",
        "        state[\"preview\"] = line(state[\"first\"], end) if end else []\n",
        "        refresh()                        # highlights the path while you drag\n",
        "\n",
        "\n",
        "# Checks the selected line when mouse is released.\n",
        "def release(e):\n",
        "    if state[\"running\"] and state[\"first\"] is not None:\n",
        "        end = cell_at(e)\n",
        "        if end and end != state[\"first\"]:                 # a real drag, not a single click\n",
        "            before = len(state[\"done\"])\n",
        "            cells = line(state[\"first\"], end)\n",
        "            check_line(cells)\n",
        "            if len(state[\"done\"]) == before:              # nothing new was found\n",
        "                text = \"\".join(state[\"grid\"][y * N + x] for y, x in cells)\n",
        "                if cells and any(text in (w, w[::-1]) for w in state[\"done\"]):\n",
        "                    note.config(text=\"Already found\")     # كلمة اتلقت قبل كده: من غير خصم\n",
        "                else:\n",
        "                    note.config(text=\"Not one of the hidden words\")\n",
        "                root.after(1500, lambda: note.config(text=\"\"))\n",
        "    state[\"first\"], state[\"preview\"] = None, []\n",
        "    refresh()\n",
        "\n",
        "\n",
        "# Checks if selected letters match a hidden word.\n",
        "def check_line(cells):\n",
        "    text = \"\".join(state[\"grid\"][y * N + x] for y, x in cells)\n",
        "    for w in state[\"words\"]:\n",
        "        if w not in state[\"done\"] and text in (w, w[::-1]):\n",
        "            color = WORD_COLORS[len(state[\"done\"]) % len(WORD_COLORS)]\n",
        "            state[\"done\"][w] = color\n",
        "            state[\"found\"].update(cells)\n",
        "            state[\"found_lines\"].append((cells, color))\n",
        "            state[\"score\"] += 10 * len(w)\n",
        "    if len(state[\"done\"]) == len(state[\"words\"]):\n",
        "        state[\"running\"] = False\n",
        "        state[\"score\"] += state[\"left\"]            # time bonus\n",
        "        msg.config(text=\"YOU WIN!\", fg=GREEN_DARK)\n",
        "        show_win()\n",
        "\n",
        "\n",
        "BALLOON_COLORS = [\"#f0a35e\", \"#6cbb8a\", \"#e86a6a\", \"#6ab0f0\", \"#f2d45c\", \"#b48be8\"]\n",
        "balloons = []\n",
        "\n",
        "\n",
        "# Shows the win banner and spawns floating balloons.\n",
        "def show_win():\n",
        "    \"\"\"كلمة YOU WIN كبيرة في نص اللوحة + بلالين بتطلع لفوق.\"\"\"\n",
        "    board.delete(\"win\")\n",
        "    balloons.clear()\n",
        "    for i in range(14):\n",
        "        x = random.randint(30, BOARD_W - 30)\n",
        "        y = random.randint(BOARD_W + 10, BOARD_W + 260)\n",
        "        tag, col = \"bal%d\" % i, random.choice(BALLOON_COLORS)\n",
        "        board.create_line(x, y + 22, x, y + 75, fill=MUTED, tags=(\"win\", tag))           # الخيط\n",
        "        board.create_polygon(x - 4, y + 28, x + 4, y + 28, x, y + 21, fill=col,\n",
        "                             outline=col, tags=(\"win\", tag))                              # العقدة\n",
        "        board.create_oval(x - 17, y - 22, x + 17, y + 22, fill=col, outline=col,\n",
        "                          tags=(\"win\", tag))                                              # البالونة\n",
        "        board.create_oval(x - 10, y - 15, x - 5, y - 7, fill=WHITE, outline=WHITE,\n",
        "                          tags=(\"win\", tag))                                              # لمعة\n",
        "        balloons.append([tag, random.uniform(1.5, 3.5), random.uniform(0, 6.28), y])\n",
        "    mid = BOARD_W / 2\n",
        "    rrect(board, 50, mid - 85, BOARD_W - 50, mid + 85, 24, fill=BG, outline=GREEN,\n",
        "          width=3, tags=\"win\")\n",
        "    board.create_text(mid, mid - 25, text=\"YOU WIN!\", font=F_WIN, fill=GREEN_DARK, tags=\"win\")\n",
        "    board.create_text(mid, mid + 45, text=\"Score: %d\" % state[\"score\"],\n",
        "                      font=(FONT, 16, \"bold\"), fill=INK, tags=\"win\")\n",
        "    animate_balloons()\n",
        "\n",
        "\n",
        "# Moves the celebration balloons upward across the screen.\n",
        "def animate_balloons():\n",
        "    for b in balloons:\n",
        "        b[2] += 0.08\n",
        "        board.move(b[0], math.sin(b[2]) * 1.2, -b[1])\n",
        "        b[3] -= b[1]\n",
        "        if b[3] < -85:                                 # طلعت برا: ترجع من تحت\n",
        "            board.move(b[0], 0, BOARD_W + 200)\n",
        "            b[3] += BOARD_W + 200\n",
        "    state[\"bjob\"] = root.after(30, animate_balloons)\n",
        "\n",
        "\n",
        "# Handles mouse click events on the game board.\n",
        "def on_press(e):\n",
        "    cell = cell_at(e)\n",
        "    if cell:\n",
        "        press(*cell)\n",
        "\n",
        "\n",
        "# Highlights the tile currently under the mouse cursor.\n",
        "def on_hover(e):\n",
        "    cell = cell_at(e)\n",
        "    if cell != state[\"hover\"]:\n",
        "        state[\"hover\"] = cell\n",
        "        refresh()\n",
        "\n",
        "\n",
        "# Clears tile hover highlight when mouse leaves board.\n",
        "def on_leave(e):\n",
        "    state[\"hover\"] = None\n",
        "    refresh()\n",
        "\n",
        "\n",
        "board.bind(\"<ButtonPress-1>\", on_press)\n",
        "board.bind(\"<B1-Motion>\", motion)\n",
        "board.bind(\"<ButtonRelease-1>\", release)\n",
        "board.bind(\"<Motion>\", on_hover)\n",
        "board.bind(\"<Leave>\", on_leave)\n",
        "\n",
        "root.update_idletasks()\n",
        "new_game()\n",
        "root.mainloop()"
      ]
    }
  ],
  "metadata": {
    "kernelspec": {
      "display_name": "Python 3",
      "language": "python",
      "name": "python3"
    },
    "language_info": {
      "codemirror_mode": {
        "name": "ipython",
        "version": 3
      },
      "file_extension": ".py",
      "mimetype": "text/x-python",
      "name": "python",
      "nbconvert_exporter": "python",
      "pygments_lexer": "ipython3",
      "version": "3.13.14"
    }
  },
  "nbformat": 4,
  "nbformat_minor": 5
}


NameError: name 'null' is not defined